# Kvasir-VQA x1 — BLIP-2 VQA fine-tuning

Fine-tune a stronger VQA transformer (BLIP-2 / InstructBLIP) on the x1 splits. The notebook mirrors the BLIP baseline but upgrades the backbone and keeps outputs under `2_modeling/06_blip2_finetune/out/`.

In [9]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset

from transformers import Blip2Processor, Blip2ForConditionalGeneration
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import evaluate

In [10]:
# Paths & config

def find_dataset_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "0_dataset_prep").exists():
            return p
    raise RuntimeError(f"Could not locate dataset root containing '0_dataset_prep'. cwd={start}")

DATA_ROOT = find_dataset_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
OUT_DIR = DATA_ROOT / "2_modeling" / "06_blip2_finetune" / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Swap to another backbone if you prefer (e.g., 'Salesforce/instructblip-vicuna-7b' for instruction-tuned VQA)
MODEL_NAME = "Salesforce/blip2-flan-t5-xl"
USE_8BIT = True  # set False if bitsandbytes not available
DEVICE_MAP = "auto"  # use HF accelerate device placement; set None to disable
TORCH_DTYPE = torch.float16
PROMPT_TEMPLATE = "Question: {question}\nAnswer:"

SEED = 42
MAX_ANSWER_LEN = 16
QUESTION_MAX_LEN = 64
MAX_GEN_TOKENS = 12


LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q", "k", "v", "o", "wi_0", "wi_1", "wo", "query", "key", "value"]
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
NUM_EPOCHS = 1
LR = 1e-5

MAX_TRAIN_SAMPLES = None  # set int for smoke tests
MAX_VAL_SAMPLES = None
MAX_TEST_SAMPLES = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("Data root:", DATA_ROOT)
print("Metadata:", META_CSV)
print("Out dir:", OUT_DIR)
print("Device:", DEVICE)


Data root: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Metadata: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv
Out dir: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/06_blip2_finetune/out
Device: cuda


In [11]:
# Load metadata
meta = pd.read_csv(META_CSV)

images_base = DATA_ROOT / "0_dataset_prep"
meta["image_path"] = meta["image_path"].apply(
    lambda p: str((images_base / p).resolve()) if not Path(p).is_absolute() else p
)

if "split" not in meta.columns:
    raise RuntimeError("Missing 'split' column. Run dataset prep split step first.")

train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df = meta[meta["split"] == "validation"].reset_index(drop=True)
test_df = meta[meta["split"] == "test"].reset_index(drop=True)

# Drop rows missing answers
train_df = train_df.dropna(subset=["answer"]).reset_index(drop=True)
val_df = val_df.dropna(subset=["answer"]).reset_index(drop=True)
test_df = test_df.dropna(subset=["answer"]).reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})


{'train': 46923, 'val': 5928, 'test': 5947}


In [12]:
# Optional subsampling
if MAX_TRAIN_SAMPLES is not None:
    train_df = train_df.sample(min(MAX_TRAIN_SAMPLES, len(train_df)), random_state=SEED).reset_index(drop=True)
if MAX_VAL_SAMPLES is not None:
    val_df = val_df.sample(min(MAX_VAL_SAMPLES, len(val_df)), random_state=SEED).reset_index(drop=True)
if MAX_TEST_SAMPLES is not None:
    test_df = test_df.sample(min(MAX_TEST_SAMPLES, len(test_df)), random_state=SEED).reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})

{'train': 46923, 'val': 5928, 'test': 5947}


In [13]:
# Load processor + model
processor = Blip2Processor.from_pretrained(MODEL_NAME)

bnb_config = None
if USE_8BIT:
    try:
        from transformers import BitsAndBytesConfig
        bnb_config = BitsAndBytesConfig(
            load_in_8bit=True,
            llm_int8_enable_fp32_cpu_offload=True,
        )
    except Exception as e:
        print("8-bit quantization unavailable, falling back to full precision:", e)

if bnb_config is not None:
    model = Blip2ForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map=DEVICE_MAP,
    )
    # prepare for k-bit training and add LoRA adapters so the quantized model is trainable
    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="SEQ_2_SEQ_LM",
    )
    model = get_peft_model(model, lora_config)
else:
    model = Blip2ForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=TORCH_DTYPE,
        device_map=DEVICE_MAP if DEVICE_MAP is not None else None,
    )
    if DEVICE_MAP is None:
        model.to(DEVICE)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "right"
model.config.text_config.pad_token_id = processor.tokenizer.pad_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.use_cache = False

try:
    model.print_trainable_parameters()
except Exception:
    pass


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 18.00 MiB. GPU 0 has a total capacity of 15.47 GiB of which 26.81 MiB is free. Process 84356 has 7.83 GiB memory in use. Process 301391 has 4.70 GiB memory in use. Including non-PyTorch memory, this process has 2.18 GiB memory in use. Of the allocated memory 1.73 GiB is allocated by PyTorch, and 177.49 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
class VQADataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        prompt = PROMPT_TEMPLATE.format(question=str(row["question"]))
        inputs = processor(images=image, text=prompt, return_tensors="pt", padding="max_length", truncation=True, max_length=QUESTION_MAX_LEN)
        labels = processor.tokenizer(
            str(row["answer"]) if pd.notna(row["answer"]) else "",
            max_length=MAX_ANSWER_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        ).input_ids.squeeze(0)
        labels[labels == processor.tokenizer.pad_token_id] = -100
        item = {k: v.squeeze(0) for k, v in inputs.items()}
        item["labels"] = labels
        return item

def collate_fn(batch):
    keys = batch[0].keys()
    return {k: torch.stack([b[k] for b in batch]) for k in keys}

train_ds = VQADataset(train_df)
val_ds = VQADataset(val_df)
test_ds = VQADataset(test_df)

print("Sample prompt:", PROMPT_TEMPLATE.format(question=train_df.iloc[0]["question"]))

Sample prompt: Question: Are there any abnormalities in the image? Check all that are present.
Answer:


In [ ]:
class VQATrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Drop HF's helper arg that T5 models don't accept
        inputs = dict(inputs)
        inputs.pop("num_items_in_batch", None)
        labels = inputs.pop("labels", None)
        outputs = model(**inputs, labels=labels)
        loss = outputs.loss if hasattr(outputs, "loss") else outputs["loss"]
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir=str(OUT_DIR / "checkpoints"),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    remove_unused_columns=False,
    report_to=[],
)

trainer = VQATrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
)

print(trainer)


No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:

trainer.train()

# Save final model
final_dir = OUT_DIR / "final_model"
trainer.save_model(final_dir)
processor.save_pretrained(final_dir)
print("Saved model to", final_dir)

Expanding inputs for image tokens in BLIP-2 should be done in processing. Please follow instruction here (https://gist.github.com/zucchini-nlp/e9f20b054fa322f84ac9311d9ab67042) to update your BLIP-2 model. Using processors without these attributes in the config is deprecated and will throw an error in v4.50.
/home/aristotle/anaconda3/envs/vqa-rag/lib/python3.11/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/aristotle/anaconda3/envs/vqa-rag/lib/python3.11/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


OutOfMemoryError: CUDA out of memory. Tried to allocate 252.00 MiB. GPU 0 has a total capacity of 15.47 GiB of which 136.62 MiB is free. Process 84356 has 7.83 GiB memory in use. Process 301391 has 4.70 GiB memory in use. Including non-PyTorch memory, this process has 2.03 GiB memory in use. Of the allocated memory 1.61 GiB is allocated by PyTorch, and 149.98 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Evaluate with BLEU/ROUGE on test split
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

model.eval()

def generate_answer(row):
    img = Image.open(row["image_path"]).convert("RGB")
    prompt = PROMPT_TEMPLATE.format(question=str(row["question"]))
    inputs = processor(images=img, text=prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_GEN_TOKENS)
    return processor.tokenizer.decode(out[0], skip_special_tokens=True).strip()

preds = []
refs = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="BLIP-2 eval"):
    preds.append(generate_answer(row))
    refs.append(str(row["answer"]))

bleu_refs = [[r] for r in refs]
bleu_score = bleu.compute(predictions=preds, references=bleu_refs)["bleu"]
rouge_l = rouge.compute(predictions=preds, references=refs)["rougeL"]

results = {"bleu": bleu_score, "rougeL": rouge_l}
with open(OUT_DIR / "metrics_test.json", "w") as f:
    json.dump(results, f, indent=2)

print(results)